In [49]:
import numpy as np 
import pandas as pd

In [50]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, weights=[0.9, 0.1], flip_y=0, random_state=42)

In [51]:
np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# model 1

In [53]:
from sklearn.linear_model import LogisticRegression

# Train the model
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)

# Predict on the test set
pred = log_reg.predict(X_test)

In [54]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



# model 2

In [55]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators= 30, max_depth=3)

rf_clf.fit(X_train, y_train)

pred = rf_clf.predict(X_test)

In [56]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.96      0.99      0.98       270
           1       0.91      0.67      0.77        30

    accuracy                           0.96       300
   macro avg       0.94      0.83      0.87       300
weighted avg       0.96      0.96      0.96       300



# model 3

In [57]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgb_clf.fit(X_train, y_train)

pred = xgb_clf.predict(X_test)

In [58]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



# model 4

imbalance classification

In [59]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)

smt_X_train, smt_y_train = smt.fit_resample(X_train, y_train)

In [60]:
np.unique(smt_y_train, return_counts=True)

(array([0, 1]), array([619, 619]))

In [61]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgb_clf.fit(smt_X_train, smt_y_train)

pred = xgb_clf.predict(X_test)

In [62]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



# mlflow 1

In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek

In [64]:
models = [
    (
        "Logistic Regression",
        LogisticRegression(C=1, solver='liblinear'),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest",
        RandomForestClassifier(n_estimators=30, max_depth=3),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        (smt_X_train, smt_y_train),
        (X_test, y_test)
    )
]

In [66]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    
    X_test = test_set[0]
    y_test = test_set[1]

    model.fit(X_train, y_train)
    
    pred = model.predict(X_test)
    
    report = classification_report(y_test, pred, output_dict=True)
    reports.append(report)

In [68]:
# reports

In [70]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_experiment("Second Experiment")
mlflow.set_tracking_uri(uri='http://127.0.0.1:5000/')

for i, element in enumerate(models):
    model_name = element[0]

    model = element[1]

    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model', model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")          


2026/03/11 10:57:43 INFO mlflow.tracking.fluent: Experiment with name 'Second Experiment' does not exist. Creating a new experiment.
2026/03/11 10:57:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 10:57:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/6/runs/11790a5e1d554ee39e947f93f6de4021
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


2026/03/11 10:58:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 10:58:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/6/runs/4bec97f701fb490191645f74061f3eed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


2026/03/11 10:58:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/6/runs/96b53e35ed7948749db59cdfcee63972
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


2026/03/11 10:58:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/6/runs/1f82bd1840a0420e941d25f728f202ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


# mlflow 2

In [71]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (smt_X_train, smt_y_train),
        (X_test, y_test)
    )
]

In [72]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [74]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_experiment("Third Exprement")
mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]

    model = element[1]
    
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):  
        mlflow.log_params(params)      
        mlflow.log_param("model", model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2026/03/11 11:20:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 11:20:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/11 11:20:43 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/7/runs/9582fb8ed12b4246a201bfa9bc909358
🧪 View experiment at: http://localhost:5000/#/experiments/7


2026/03/11 11:20:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 11:20:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/11 11:20:53 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


🏃 View run Random Forest at: http://localhost:5000/#/experiments/7/runs/58e90980e3a54c52ba745ff94ceb38ba
🧪 View experiment at: http://localhost:5000/#/experiments/7


2026/03/11 11:21:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/7/runs/8a6c33e524a64c3b93f38903912fdd4a
🧪 View experiment at: http://localhost:5000/#/experiments/7


AttributeError: 'dict' object has no attribute 'save_model'

# Register the model

In [83]:
run_id = input("Enter Run ID: ")

model_name = "XGB-Smote"

model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(model_uri, model_name)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/03/11 16:57:51 WARNING mlflow.tracking._model_registry.fluent: Run with id 1f82bd1840a0420e941d25f728f202ca has no artifacts at artifact path 'model', registering model based on models:/m-b2e230e4a22045fca86a468feba7f620 instead
2026/03/11 16:57:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


# Load the model

In [84]:
model_version = 1

model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)

y_pred = loaded_model.predict(X_test)

In [ ]:
# production

# dev_model_uri = f"models:/{model_name}/{model_version}"

# prod_model = 'anomaly-detection-prod'

# client = mlflow.MlflowClient()

# client.copy_model_version(src_model_uri='', dst_name='')